# RoPE（旋转位置编码）手撕实现

## 1. 思想
对位置 $m$ 的向量 $q$，乘以旋转矩阵 $R_m$：$q_m=R_m q$。内积 $q_m^\top k_n=(R_m q)^\top(R_n k)=q^\top R_{n-m}^\top k$ **只依赖相对位置 $n-m$**，从而把绝对位置编码变成相对。

## 2. 旋转矩阵（按 2D 一组）
$$R_m=\begin{bmatrix}\cos m\theta_0&-\sin m\theta_0\\\sin m\theta_0&\cos m\theta_0\end{bmatrix}\oplus\cdots\oplus\begin{bmatrix}\cos m\theta_{d/2-1}&-\sin m\theta_{d/2-1}\\\sin m\theta_{d/2-1}&\cos m\theta_{d/2-1}\end{bmatrix}$$
$\theta_i=10000^{-2i/d}$。块对角，故可拆成"逐对旋转"高效实现。

## 3. rotate_half 高效实现
把 $x=[x_1,x_2]$（前后各半），$\text{rotate\_half}(x)=[-x_2,x_1]$，则 $R_m x \approx x\odot\cos m\theta + \text{rotate\_half}(x)\odot\sin m\theta$。
- 注意 `cos/sin` 的排列：`cat((f,f))` 对应 rotate_half 切前半/后半；`repeat_interleave(2)` 对应按偶奇位切。两种实现要对齐。

In [ ]:
import torch
import torch.nn as nn
import math

class RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_position_embeddings=2048, base=10000):
        super().__init__()
        self.dim = dim
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))   # [dim/2]
        self.register_buffer('inv_freq', inv_freq)
        self._set_cos_sin_cache(max_position_embeddings)

    def _set_cos_sin_cache(self, seq_len):
        t = torch.arange(seq_len, device=self.inv_freq.device).type_as(self.inv_freq)
        freqs = torch.outer(t, self.inv_freq)                  # [seq, dim/2]
        emb = torch.cat((freqs, freqs), dim=-1)                # [seq, dim] 对应 rotate_half
        self.register_buffer('cos_cached', emb.cos())
        self.register_buffer('sin_cached', emb.sin())

    def forward(self, seq_len):
        return self.cos_cached[:seq_len], self.sin_cached[:seq_len]

def rotate_half(x):
    x1, x2 = x[..., :x.shape[-1] // 2], x[..., x.shape[-1] // 2:]
    return torch.cat((-x2, x1), dim=-1)

def apply_rotary_pos_emb(q, k, cos, sin):
    # q,k: [B, h, T, d] ; cos,sin: [T, d] -> 广播到 [1,1,T,d]
    cos = cos.unsqueeze(0).unsqueeze(0)
    sin = sin.unsqueeze(0).unsqueeze(0)
    q_emb = q * cos + rotate_half(q) * sin
    k_emb = k * cos + rotate_half(k) * sin
    return q_emb, k_emb

In [ ]:
# 验证：1) 保持范数  2) 内积只依赖相对位置
torch.manual_seed(0)
dim, h, B = 8, 2, 1
rope = RotaryEmbedding(dim, 32)
cos, sin = rope(32)

q = torch.randn(B, h, 1, dim)               # 位置 0
k = torch.randn(B, h, 1, dim)               # 位置 0
q0, k0 = apply_rotary_pos_emb(q, k, cos[:1], sin[:1])
print('范数保持:', torch.allclose(q.norm(), q0.norm(), atol=1e-5))

# 相对位置性质：q 在位置 3、k 在位置 5 的内积 == q 在位置 0、k 在位置 2
q3, _ = apply_rotary_pos_emb(q, k, cos[3:4], sin[3:4])
_, k5 = apply_rotary_pos_emb(q, k, cos[5:6], sin[5:6])
q0b, k2 = apply_rotary_pos_emb(q, k, cos[0:1], sin[0:1]), apply_rotary_pos_emb(q, k, cos[2:3], sin[2:3])
dot_shifted = (q3 * k5).sum(-1)
q0_, k2_ = apply_rotary_pos_emb(q, k, cos[0:1], sin[0:1])
_, k2_ = apply_rotary_pos_emb(q, k, cos[2:3], sin[2:3])
dot_ref = (q0_ * k2_).sum(-1)
print('相对位置内积一致:', torch.allclose(dot_shifted, dot_ref, atol=1e-5))

## 小结 / 易错点
- 原仓库 `ROPE.ipynb` 缺 `import torch / torch.nn`，会 NameError，本版已补。
- `cat((freqs,freqs))` 配 `rotate_half`（切前后半）；若用 `repeat_interleave(2)` 则要按偶/奇位切，二者不可混用。
- RoPE 只作用在 Q、K 上，V 不旋转。
- 长上下文需调整 `base`（如 500000）以减缓高频衰减。

## ✅ 测试验证

In [ ]:
# 验证 RoPE 性质
import torch
import math

# RoPE 关键性质:
# 1. 保持向量内积（相对位置信息）
# 2. 旋转矩阵正交（不改变向量范数）
# 3. 位置 m+1 的旋转是位置 m 的复合

# 测试正交性: 旋转不改变范数
d = 8
theta = 0.5
# 构造旋转矩阵（2D 旋转块）
rot = torch.tensor([[math.cos(theta), -math.sin(theta)],
                     [math.sin(theta),  math.cos(theta)]])
x = torch.tensor([3.0, 4.0])  # norm = 5
x_rot = rot @ x
assert abs(x_rot.norm().item() - x.norm().item()) < 1e-5, \
    f"RoPE 改变了范数: {x_rot.norm()} vs {x.norm()}"
print("  ✓ RoPE 保持向量范数不变")

# 测试相对位置: <RoPE(x, m), RoPE(y, m+k)> 只依赖 k
# 简化: 2D 情况下验证
def rope_2d(x, pos, d=2):
    theta = pos * 0.5
    rot = torch.tensor([[math.cos(theta), -math.sin(theta)],
                         [math.sin(theta),  math.cos(theta)]])
    return rot @ x

x = torch.tensor([1.0, 0.0])
y = torch.tensor([0.0, 1.0])
# 位置 0 和 1 的内积
dot_01 = torch.dot(rope_2d(x, 0), rope_2d(y, 1))
# 位置 5 和 6 的内积（相对距离仍为 1）
dot_56 = torch.dot(rope_2d(x, 5), rope_2d(y, 6))
assert abs(dot_01.item() - dot_56.item()) < 1e-5, \
    f"RoPE 内积不满足相对位置: {dot_01} vs {dot_56}"
print("  ✓ RoPE 内积只依赖相对位置")

print("✅ RoPE 测试通过: 正交性、相对位置编码正确")
